## Cálculo dos KPIs: cleaned para curated
Esse notebook lê as 20 tabelas da camada `cleaned` (as 15 views do `dm` mais as 5 novas), calcula os KPIs de saúde, produção, manutenção, ativos, alertas, realocação e estoque, calcula os IS Energético e Ambiental por string e por filial, e salva o resultado na camada `curated`, que é a base final do dashboard.

#### Rotina de conexão com o Neon

In [0]:
%pip install python-dotenv

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../apollo-core/apollo_connection_neon_core

#### Configuração

##### Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

##### Camadas

In [0]:
CLEANED = f"{CATALOG}.cleaned"
CURATED = f"{CATALOG}.curated"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CURATED}")

##### Parâmetros dos KPIs

In [0]:
LIMIAR_SAUDE = 70          # definido no projeto: alerta de saúde abaixo de 70 para todas as empresas
PADRAO_MIN = 0.80          # A DEFINIR: pico/nominal mínimo para a string contar como dentro do padrão da etiqueta
QUEDA_BRUSCA_PTS = 5       # A DEFINIR: queda de saúde (em pontos) entre duas medições que conta como queda brusca
META_IS = 80               # A DEFINIR: meta do IS Geral, usada no desvio dos cards
META_RESOLUCAO_DIAS = 7    # A DEFINIR: meta do tempo médio de resolução da OS, usada no desvio dos cards

#### Conferir a cleaned

##### Tabelas que precisam existir

In [0]:
ESPERADAS = [
    "dim_company", "dim_segment", "dim_company_unit", "dim_panel_model", "dim_inverter", "dim_string",
    "dim_panel", "dim_employee", "dim_date", "dim_maintenance_type", "dim_part",
    "fact_string_daily_performance", "fact_string_health", "fact_maintenance", "fact_part_consumption",
    "fact_daily_active_user", "fact_warning", "fact_relocation", "fact_stock", "vw_maintenance_chain",
]

existentes = [t.tableName for t in spark.sql(f"SHOW TABLES IN {CLEANED}").collect()]
print("Faltando na cleaned:", [t for t in ESPERADAS if t not in existentes])

#### Leitura e gravação

##### Ler uma tabela da cleaned

In [0]:
def read_cleaned(table):
    return spark.table(f"{CLEANED}.{table}")

##### Salvar uma tabela na curated

In [0]:
def save_curated(df, table):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CURATED}.{table}")

##### Contar linhas que atendem a uma condição

In [0]:
def count_if(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

#### Tabelas de apoio

##### Filial

In [0]:
def unit_info():
    return read_cleaned("dim_company_unit").select(
        "company_unit_key", "company_key", "unit_name", "city", "state", "is_active", "kwp_total")

##### String e filial

In [0]:
def string_unit():
    strings = read_cleaned("dim_string").select("string_key", "inverter_key", "panel_model_key")
    inverters = read_cleaned("dim_inverter").select("inverter_key", "company_unit_key")
    return strings.join(inverters, "inverter_key").select("string_key", "panel_model_key", "company_unit_key")

##### Calendário

In [0]:
def calendario():
    return read_cleaned("dim_date").select(
        "date_key", "full_date", F.date_format("full_date", "yyyy-MM").alias("ano_mes"))

#### Saúde das strings

##### Saúde atual de cada string

In [0]:
# última nota de cada string; faixas do projeto: abaixo de 50 crítica, 50 a 59 alta, 60 a 69 média
def saude_atual_string():
    w = Window.partitionBy("string_key").orderBy(F.col("calculated_at").desc())
    ultima = (read_cleaned("fact_string_health")
              .withColumn("rn", F.row_number().over(w))
              .filter(F.col("rn") == 1)
              .select("string_key", "calculated_at", "health_score"))
    nota = F.col("health_score")
    faixa = (F.when(nota < 50, "CRITICA")
              .when(nota < 60, "ALTA")
              .when(nota < LIMIAR_SAUDE, "MEDIA")
              .otherwise("OK"))
    return (ultima.withColumn("faixa", faixa)
                  .join(string_unit().select("string_key", "company_unit_key"), "string_key"))

##### Saúde por filial

In [0]:
# strings sem medição não têm nota, então ficam fora da conta
def saude_por_filial(saude_string):
    df = saude_string.groupBy("company_unit_key").agg(
        F.count("*").alias("strings_com_nota"),
        F.round(F.avg("health_score"), 2).alias("saude_media"),
        count_if(F.col("health_score") < LIMIAR_SAUDE).alias("strings_abaixo_limiar"),
        count_if(F.col("faixa") == "CRITICA").alias("strings_criticas"),
        count_if(F.col("faixa") == "ALTA").alias("strings_altas"),
        count_if(F.col("faixa") == "MEDIA").alias("strings_medias"),
    )
    return (df.withColumn("pct_abaixo_limiar", F.round(100 * F.col("strings_abaixo_limiar") / F.col("strings_com_nota"), 2))
              .withColumn("desvio_limiar", F.round(F.col("saude_media") - LIMIAR_SAUDE, 2)))

##### Evolução diária da saúde por filial

In [0]:
def saude_diaria_filial():
    saude = read_cleaned("fact_string_health").join(string_unit().select("string_key", "company_unit_key"), "string_key")
    return (saude.join(calendario().select("date_key", "full_date"), "date_key")
                 .groupBy("company_unit_key", "full_date")
                 .agg(F.round(F.avg("health_score"), 2).alias("saude_media")))

#### Produção

##### Energia gerada por filial e mês

In [0]:
# kwh_por_kwp usa a capacidade instalada atual da filial, então é uma aproximação nos meses antigos
def producao_filial_mes():
    perf = (read_cleaned("fact_string_daily_performance")
            .join(string_unit().select("string_key", "company_unit_key"), "string_key")
            .join(calendario().select("date_key", "ano_mes"), "date_key"))
    mensal = perf.groupBy("company_unit_key", "ano_mes").agg(
        F.round(F.sum("generated_energy_kwh"), 2).alias("energia_kwh"),
        F.sum("fault_count").alias("leituras_com_falha"),
    )
    kwp = unit_info().select("company_unit_key", "kwp_total")
    return (mensal.join(kwp, "company_unit_key", "left")
                  .withColumn("kwh_por_kwp", F.when(F.col("kwp_total") > 0, F.round(F.col("energia_kwh") / F.col("kwp_total"), 2))))

#### Manutenção

##### OS por filial e mês

In [0]:
def manutencao_filial_mes():
    cadeia = read_cleaned("vw_maintenance_chain").select(
        F.col("id").alias("maintenance_key"), (F.col("chain_level") > 1).alias("retrabalho"))
    m = (read_cleaned("fact_maintenance")
         .join(calendario().select(F.col("date_key").alias("opening_date_key"), "ano_mes"), "opening_date_key")
         .join(cadeia, "maintenance_key", "left"))

    status = F.col("maintenance_status")
    com_orcamento = (status == "CONCLUIDA") & (F.col("estimated_cost") > 0)

    df = m.groupBy("company_unit_key", "ano_mes").agg(
        F.count("*").alias("total_os"),
        count_if(status == "ABERTA").alias("os_abertas"),
        count_if(status == "EM_ANDAMENTO").alias("os_em_andamento"),
        count_if(status == "CONCLUIDA").alias("os_concluidas"),
        count_if(status == "CANCELADA").alias("os_canceladas"),
        count_if(F.col("calculated_status") == "ATRASADA").alias("os_atrasadas"),
        count_if(F.coalesce(F.col("retrabalho"), F.lit(False))).alias("os_retrabalho"),
        F.round(F.avg("resolution_days"), 2).alias("tempo_medio_resolucao_dias"),
        F.round(F.sum("total_cost"), 2).alias("custo_total"),
        F.sum(F.when(com_orcamento, F.col("estimated_cost"))).alias("orcado_concluidas"),
        F.sum(F.when(com_orcamento, F.col("total_cost"))).alias("realizado_concluidas"),
    )
    return (df.withColumn("taxa_cancelamento", F.round(100 * F.col("os_canceladas") / F.col("total_os"), 2))
              .withColumn("taxa_retrabalho", F.round(100 * F.col("os_retrabalho") / F.col("total_os"), 2))
              .withColumn("desvio_orcamento_pct", F.when(F.col("orcado_concluidas") > 0,
                          F.round(100 * (F.col("realizado_concluidas") - F.col("orcado_concluidas")) / F.col("orcado_concluidas"), 2)))
              .withColumn("desvio_meta_resolucao", F.round(F.col("tempo_medio_resolucao_dias") - META_RESOLUCAO_DIAS, 2)))

##### OS por tipo de manutenção e tipo de alerta

In [0]:
def os_por_tipo():
    tipos = read_cleaned("dim_maintenance_type").select(
        "maintenance_type_key", F.col("type_name").alias("tipo_manutencao"))
    m = (read_cleaned("fact_maintenance")
         .join(tipos, "maintenance_type_key")
         .join(calendario().select(F.col("date_key").alias("opening_date_key"), "ano_mes"), "opening_date_key"))
    return (m.groupBy("company_unit_key", "ano_mes", "tipo_manutencao", "warning_type", "maintenance_status")
             .agg(F.count("*").alias("qtd_os")))

#### Ativos

##### Placas por status e filial

In [0]:
def ativos_filial():
    placas = read_cleaned("dim_panel").join(string_unit().select("string_key", "company_unit_key"), "string_key")
    status = F.col("operating_stats")
    df = placas.groupBy("company_unit_key").agg(
        F.count("*").alias("total_placas"),
        count_if(status == "EM_ESTOQUE").alias("placas_em_estoque"),
        count_if(status == "OPERACIONAL").alias("placas_operacionais"),
        count_if(status == "MANUTENCAO").alias("placas_em_manutencao"),
        count_if(status == "INATIVO").alias("placas_inativas"),
        count_if(status == "FALHA").alias("placas_em_falha"),
    )
    df = df.join(unit_info().select("company_unit_key", "kwp_total"), "company_unit_key", "left")
    return (df.withColumn("pct_falha_ou_manutencao",
                          F.round(100 * (F.col("placas_em_falha") + F.col("placas_em_manutencao")) / F.col("total_placas"), 2))
              .withColumn("taxa_ativacao",
                          F.round(100 * (F.col("total_placas") - F.col("placas_em_estoque")) / F.col("total_placas"), 2)))

#### Alertas

##### Alertas por filial

In [0]:
# alerta sem OS = nunca virou OS, ou a única OS foi cancelada, e ainda não foi resolvido
def alertas_filial():
    w = read_cleaned("fact_warning")
    sem_os = F.col("maintenance_key").isNull() | (F.col("maintenance_status") == "CANCELADA")
    return w.groupBy("company_unit_key").agg(
        F.count("*").alias("total_alertas"),
        count_if(F.col("status") == "ATIVO").alias("alertas_ativos"),
        count_if(F.col("status") == "RECONHECIDO").alias("alertas_reconhecidos"),
        count_if(F.col("status") == "RESOLVIDO").alias("alertas_resolvidos"),
        count_if(F.col("severity") == "CRITICA").alias("alertas_criticos"),
        count_if(sem_os & (F.col("status") != "RESOLVIDO")).alias("alertas_sem_os"),
        F.round(F.avg("hours_to_maintenance"), 2).alias("horas_medias_ate_os"),
        F.round(F.avg("resolution_days"), 2).alias("dias_medios_resolucao_alerta"),
    )

#### Realocação e estoque

##### Realocações por filial de origem

In [0]:
def realocacao_filial():
    r = read_cleaned("fact_relocation")
    return r.groupBy("origin_unit_key", "relocation_scope").agg(
        F.count("*").alias("total_solicitacoes"),
        count_if(F.col("status") == "PENDENTE").alias("pendentes"),
        count_if(F.col("status") == "APROVADA").alias("aprovadas"),
        count_if(F.col("status") == "REJEITADA").alias("rejeitadas"),
        count_if(F.col("status") == "CONCLUIDA").alias("concluidas"),
        F.sum("panel_quantity").alias("placas_envolvidas"),
        F.round(F.avg("review_days"), 2).alias("dias_medios_revisao"),
    ).withColumnRenamed("origin_unit_key", "company_unit_key")

##### Estoque por peça

In [0]:
def estoque_peca():
    pecas = read_cleaned("dim_part").select("part_key", "sku", "part_name")
    return read_cleaned("fact_stock").join(pecas, "part_key")

##### Estoque por filial

In [0]:
def estoque_filial(estoque):
    return estoque.groupBy("company_unit_key").agg(
        F.count("*").alias("pecas_cadastradas"),
        count_if(F.col("below_minimum")).alias("pecas_abaixo_minimo"),
        F.round(F.sum("stock_value"), 2).alias("valor_estoque"),
    )

#### Índices de sustentabilidade (IS)

O IS Financeiro fica de fora: ele depende de consumo e tarifa do usuário, que não existem no banco. O IS Geral é a média dos IS que existem (Energético e Ambiental).

##### IS Energético por string

In [0]:
# média de dois percentuais: dias com pico/nominal dentro do padrão da etiqueta e medições sem queda brusca de saúde
def is_energetico_string():
    no_padrao = (read_cleaned("fact_string_daily_performance")
                 .groupBy("string_key")
                 .agg(F.avg((F.col("peak_to_nominal_ratio") >= PADRAO_MIN).cast("int")).alias("pct_no_padrao")))
    sem_queda = (read_cleaned("fact_string_health")
                 .filter(F.col("health_delta").isNotNull())
                 .groupBy("string_key")
                 .agg(F.avg((F.col("health_delta") > -QUEDA_BRUSCA_PTS).cast("int")).alias("pct_sem_queda")))
    df = no_padrao.join(sem_queda, "string_key", "outer")
    media = (F.coalesce(F.col("pct_no_padrao"), F.col("pct_sem_queda")) + F.coalesce(F.col("pct_sem_queda"), F.col("pct_no_padrao"))) / 2
    return df.withColumn("is_energetico", F.round(100 * media, 2))

##### IS Ambiental por string

In [0]:
# vida útil = garantia do modelo de placa (warranty_years); idade = média das placas instaladas e ainda ativas da string
# sem garantia cadastrada o IS fica nulo (nunca um valor padrão)
def is_ambiental_string():
    placas = read_cleaned("dim_panel").filter(F.col("installation_dt").isNotNull() & F.col("deactivated_at").isNull())
    idade = placas.groupBy("string_key").agg(
        F.avg(F.datediff(F.current_date(), F.col("installation_dt")) / 365.25).alias("idade_media_anos"))
    garantia = string_unit().select("string_key", "panel_model_key").join(
        read_cleaned("dim_panel_model").select("panel_model_key", "warranty_years"), "panel_model_key")
    df = idade.join(garantia, "string_key")
    restante = (1 - F.col("idade_media_anos") / F.col("warranty_years")) * 100
    is_amb = F.when(F.col("warranty_years") > 0, F.greatest(F.lit(0.0), F.least(F.lit(100.0), restante)))
    return df.select("string_key", "idade_media_anos", "warranty_years", F.round(is_amb, 2).alias("is_ambiental"))

##### IS por string

In [0]:
def is_por_string():
    base = string_unit().select("string_key", "company_unit_key")
    energetico = is_energetico_string().select("string_key", "is_energetico")
    ambiental = is_ambiental_string().select("string_key", "is_ambiental")
    return base.join(energetico, "string_key", "left").join(ambiental, "string_key", "left")

##### IS por filial e IS Geral

In [0]:
def is_por_filial(is_string):
    df = is_string.groupBy("company_unit_key").agg(
        F.round(F.avg("is_energetico"), 2).alias("is_energetico"),
        F.round(F.avg("is_ambiental"), 2).alias("is_ambiental"),
    )
    geral = (F.coalesce(F.col("is_energetico"), F.col("is_ambiental")) + F.coalesce(F.col("is_ambiental"), F.col("is_energetico"))) / 2
    df = (df.withColumn("is_geral", F.round(geral, 2))
            .withColumn("desvio_meta", F.round(F.col("is_geral") - META_IS, 2)))
    return df.join(unit_info().select("company_unit_key", "company_key", "unit_name"), "company_unit_key")

#### Mapa de filiais

##### Filial com cidade, estado e indicadores de risco

In [0]:
# a dim_company_unit só tem cidade e estado (sem latitude e longitude), então o mapa é por cidade ou estado
def filial_mapa(saude_filial, alertas):
    return (unit_info()
            .join(saude_filial.select("company_unit_key", "saude_media", "strings_abaixo_limiar"), "company_unit_key", "left")
            .join(alertas.select("company_unit_key", "alertas_ativos"), "company_unit_key", "left")
            .fillna(0, subset=["strings_abaixo_limiar", "alertas_ativos"]))

#### Execução

##### Calcular os KPIs

In [0]:
saude_string = saude_atual_string()
saude_filial = saude_por_filial(saude_string)
alertas = alertas_filial()
estoque = estoque_peca()
is_string = is_por_string()

tabelas = {
    "kpi_saude_string": saude_string,
    "kpi_saude_filial": saude_filial,
    "kpi_saude_diaria_filial": saude_diaria_filial(),
    "kpi_producao_filial_mes": producao_filial_mes(),
    "kpi_manutencao_filial_mes": manutencao_filial_mes(),
    "kpi_os_por_tipo": os_por_tipo(),
    "kpi_ativos_filial": ativos_filial(),
    "kpi_alertas_filial": alertas,
    "kpi_realocacao_filial": realocacao_filial(),
    "kpi_estoque_peca": estoque,
    "kpi_estoque_filial": estoque_filial(estoque),
    "kpi_is_string": is_string,
    "kpi_is_filial": is_por_filial(is_string),
    "dim_filial_mapa": filial_mapa(saude_filial, alertas),
}

##### Percentis para ajudar a definir as metas

In [0]:
display(read_cleaned("fact_maintenance").select(
    F.percentile_approx("resolution_days", [0.25, 0.5, 0.75, 0.9]).alias("percentis_resolucao_dias")))
display(saude_string.select(
    F.percentile_approx("health_score", [0.1, 0.25, 0.5, 0.75]).alias("percentis_saude")))

##### Conferir o resultado

In [0]:
display(tabelas["kpi_is_filial"])

##### Salvar na curated

In [0]:
for nome, df in tabelas.items():
    save_curated(df, nome)